# 运行环境导入

In [2]:
# 启用自动重载扩展，使得在修改Python模块后不需要重启内核就能使用最新的代码
%load_ext autoreload

# 设置自动重载模式为2，表示所有模块都会被自动重载
# 这在开发过程中非常有用，因为不需要手动重启kernel来应用代码更改
%autoreload 2

import torch
from shepherd.lightning_module import LightningModule

import pickle

import rdkit
import numpy as np

from shepherd.shepherd_score_utils.conformer_generation import update_mol_coordinates

from shepherd.shepherd_score_utils.generate_point_cloud import (
    get_atomic_vdw_radii, 
    get_molecular_surface,
    get_electrostatics_given_point_charges,
)
from shepherd.shepherd_score_utils.pharm_utils.pharmacophore import get_pharmacophores

from tqdm import tqdm
from shepherd.shepherd_score_utils.pharm_utils.pharmacophore import get_pharmacophores

import json
import numpy as np

from shepherd.inference import *
import os



import json
import numpy as np

import json
import numpy as np
import torch  # 导入 torch 库，因为我们需要处理 Tensor 类型

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

from shepherd_score.evaluations.evaluate import ConfEval, UnconditionalEvalPipeline
from shepherd_score.evaluations.evaluate import ConsistencyEvalPipeline, ConditionalEvalPipeline

from shepherd_score.container import Molecule

from shepherd.extract import create_rdkit_molecule_from_mol
from shepherd_score.conformer_generation import embed_conformer_from_smiles



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# 模型参数、sample参考分子 加载

In [4]:
# 配置

chkpt = '/home1/zhh/workspace/SPD/evaluation/ckpt/last_27epoch.ckpt'
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model_pl = LightningModule.load_from_checkpoint(chkpt) #
params = model_pl.params
model_pl.to(device)
model_pl.model.device = device


with open('/home1/zhh/workspace/SPD/data/conformers/np/molblock_charges_NPs.pkl', 'rb') as f:
    # 从pkl文件中读取molblock和charges数据
    molblocks_and_charges = pickle.load(f)
    # 打印数据长度以确认实际包含的分子数量
    print(f"加载的数据包含 {len(molblocks_and_charges)} 个分子")

# ==================== 修改：处理所有天然产物分子 ====================
# 将处理所有3个天然产物分子（index 0, 1, 2）
# 每个分子将生成20个样本（batch_size=5，循环4次）

print("将对所有天然产物分子进行采样：")
for idx in range(len(molblocks_and_charges)):
    mol = rdkit.Chem.MolFromMolBlock(molblocks_and_charges[idx][0], removeHs=False)
    print(f"  - 分子 {idx}: {mol.GetNumAtoms()} 个原子")


正在以非严格模式加载模型状态，将忽略检查点中不匹配的键...
加载的数据包含 3 个分子
将对所有天然产物分子进行采样：
  - 分子 0: 78 个原子
  - 分子 1: 50 个原子
  - 分子 2: 85 个原子


In [7]:
# ==================== 计算边际分布（从原Cell 7移至此处） ====================
# 必须在采样前计算，为扩散模型提供先验分布

print("初始化特征计数器...")

# 特征类型定义
atom_types_x1 = [None, 'H', 'C', 'N', 'O', 'F', 'Cl', 'Br', 'I', 'S', 'P', 'Si']
bond_types_x1 = [None, 'SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC']
max_node_types_x4 = 10

# 初始化计数器
atom_counts = torch.zeros(len(atom_types_x1), dtype=torch.float)
bond_counts = torch.zeros(len(bond_types_x1), dtype=torch.float)
pharm_counts = torch.zeros(max_node_types_x4, dtype=torch.float)

def get_bond_type_str(bond):
    return str(bond.GetBondType())

# 统计特征出现次数
for mol_block, _ in tqdm(molblocks_and_charges, desc="计算边际分布"):
    mol = rdkit.Chem.MolFromMolBlock(mol_block, removeHs=False)
    if not mol:
        print("Warning: Failed to create molecule from MolBlock")
        continue
    
    # 统计原子类型
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        if symbol in atom_types_x1:
            atom_counts[atom_types_x1.index(symbol)] += 1
    
    # 统计键类型
    for bond in mol.GetBonds():
        bond_str = get_bond_type_str(bond)
        if bond_str in bond_types_x1:
            bond_counts[bond_types_x1.index(bond_str)] += 1
    
    # 统计药效团类型
    try:
        pharm_types_temp, _, _ = get_pharmacophores(
            mol, 
            multi_vector=False,
            check_access=False
        )
        for p_type in (pharm_types_temp + 1):
            if p_type < max_node_types_x4:
                pharm_counts[p_type] += 1
    except Exception as e:
        print(f"Warning: Could not get pharmacophores. Error: {e}")

# 归一化为概率分布
atom_marginals_x1 = (atom_counts / atom_counts.sum()) if atom_counts.sum() > 0 else torch.ones_like(atom_counts) / len(atom_counts)
bond_marginals_x1 = (bond_counts / bond_counts.sum()) if bond_counts.sum() > 0 else torch.ones_like(bond_counts) / len(bond_counts)
pharm_marginals_x4 = (pharm_counts / pharm_counts.sum()) if pharm_counts.sum() > 0 else torch.ones_like(pharm_counts) / len(pharm_counts)

print("\n✅ 边际分布计算完成")
print(f"  - Atom Marginals: {atom_marginals_x1.shape}")
print(f"  - Bond Marginals: {bond_marginals_x1.shape}")
print(f"  - Pharmacophore Marginals: {pharm_marginals_x4.shape}")

print(f"  - Atom Marginals: {atom_marginals_x1}")
print(f"  - Bond Marginals: {bond_marginals_x1}")
print(f"  - Pharmacophore Marginals: {pharm_marginals_x4}")


初始化特征计数器...


计算边际分布: 100%|██████████| 3/3 [00:00<00:00, 88.48it/s]


✅ 边际分布计算完成
  - Atom Marginals: torch.Size([12])
  - Bond Marginals: torch.Size([5])
  - Pharmacophore Marginals: torch.Size([10])
  - Atom Marginals: tensor([0.0000, 0.4742, 0.3944, 0.0047, 0.1268, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000])
  - Bond Marginals: tensor([0.0000, 0.8705, 0.0804, 0.0000, 0.0491])
  - Pharmacophore Marginals: tensor([0.0000, 0.5000, 0.1154, 0.0385, 0.3462, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000])


In [ ]:
! CUDA_VISIBLE_DEVICES="1,2"

# 对每个分子循环采样


In [ ]:
# ==================== 对每个分子循环采样（batch_size=5, 共20个样本/分子） ====================

# 辅助函数：转换数据为JSON格式
def convert_for_json(obj):
    """递归转换numpy数组和torch张量为Python列表"""
    if isinstance(obj, dict):
        return {k: convert_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [convert_for_json(elem) for elem in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, torch.Tensor):
        return obj.cpu().numpy().tolist()
    return obj

# 存储所有分子的所有样本
all_generated_samples = []

# 对每个天然产物分子进行处理
for mol_index in range(len(molblocks_and_charges)):
    print(f"\n{'='*60}")
    print(f"🔬 开始处理分子 {mol_index + 1}/{len(molblocks_and_charges)}")
    print(f"{'='*60}")
    
    # 从molblock创建RDKit分子对象
    mol = rdkit.Chem.MolFromMolBlock(molblocks_and_charges[mol_index][0], removeHs=False)
    charges = np.array(molblocks_and_charges[mol_index][1])
    
    print(f"分子信息: {mol.GetNumAtoms()} 个原子")
    display(mol)
    
    # 分子坐标标准化
    mol_coordinates = np.array(mol.GetConformer().GetPositions())
    mol_coordinates = mol_coordinates - np.mean(mol_coordinates, axis=0)
    mol = update_mol_coordinates(mol, mol_coordinates)
    
    # 条件特征提取
    centers = mol.GetConformer().GetPositions()
    radii = get_atomic_vdw_radii(mol)
    
    # 生成分子表面点云
    surface = get_molecular_surface(
        centers, 
        radii, 
        params['dataset']['x3']['num_points'],
        probe_radius=params['dataset']['probe_radius'],
        num_samples_per_atom=20,
    )
    
    # 提取药效团特征
    pharm_types, pharm_pos, pharm_direction = get_pharmacophores(
        mol,
        multi_vector=params['dataset']['x4']['multivectors'],
        check_access=params['dataset']['x4']['check_accessibility'],
    )
    
    # 计算表面静电势
    electrostatics = get_electrostatics_given_point_charges(
        charges, centers, surface,
    )
    
    # 采样参数配置
    n_atoms = 70
    batch_size = 4
    num_pharmacophores = len(pharm_types)
    num_iterations = 5  # 循环5次，每次2个，总共10个样本
    
    print(f"\n📊 采样配置:")
    print(f"  - 原子数: {n_atoms}")
    print(f"  - Batch size: {batch_size}")
    print(f"  - 迭代次数: {num_iterations}")
    print(f"  - 总样本数: {batch_size * num_iterations}")
    print(f"  - 药效团数: {num_pharmacophores}")
    
    # 循环生成20个样本（4次 × 5个/次）
    mol_samples = []
    for iteration in range(num_iterations):
        print(f"\n🔄 迭代 {iteration + 1}/{num_iterations} (生成样本 {iteration*batch_size+1}-{(iteration+1)*batch_size})...")
        
        # 调用推理采样
        generated_samples = inference_sample(
            model_pl,
            batch_size=batch_size,
            N_x1=n_atoms,
            N_x4=num_pharmacophores,
            unconditional=False,
            
            # 噪声控制
            prior_noise_scale=1.0,
            denoising_noise_scale=1.0,
            inject_noise_at_ts=[],
            inject_noise_scales=[],
            
            # 谐波化
            harmonize=False,
            harmonize_ts=[],
            harmonize_jumps=[],
            
            # 条件修复
            inpaint_x2_pos=False,
            inpaint_x3_pos=False,
            inpaint_x3_x=False,
            inpaint_x4_pos=True,
            inpaint_x4_direction=True,
            inpaint_x4_type=True,
            
            # 修复控制
            stop_inpainting_at_time_x2=0.0,
            add_noise_to_inpainted_x2_pos=0.0,
            stop_inpainting_at_time_x3=0.0,
            add_noise_to_inpainted_x3_pos=0.0,
            add_noise_to_inpainted_x3_x=0.0,
            stop_inpainting_at_time_x4=0.0,
            add_noise_to_inpainted_x4_pos=0.0,
            add_noise_to_inpainted_x4_direction=0.0,
            add_noise_to_inpainted_x4_type=0.0,
            
            # 条件输入
            center_of_mass=np.zeros(3),
            surface=surface,
            electrostatics=electrostatics,
            pharm_types=pharm_types,
            pharm_pos=pharm_pos,
            pharm_direction=pharm_direction,
            
            # 边际分布（从Cell 5获取）
            atom_marginals=atom_marginals_x1,
            bond_marginals=bond_marginals_x1,
        )
        
        # 添加分子索引信息
        for sample in generated_samples:
            sample['source_mol_index'] = mol_index
        
        mol_samples.extend(generated_samples)
        print(f"  ✓ 完成 {len(generated_samples)} 个样本")
    
    print(f"\n✅ 分子 {mol_index + 1} 采样完成: 共 {len(mol_samples)} 个样本")
    all_generated_samples.extend(mol_samples)

print(f"\n{'='*60}")
print(f"🎉 所有采样完成!")
print(f"{'='*60}")
print(f"总样本数: {len(all_generated_samples)}")
print(f"  - 分子0: {sum(1 for s in all_generated_samples if s['source_mol_index'] == 0)} 个样本")
print(f"  - 分子1: {sum(1 for s in all_generated_samples if s['source_mol_index'] == 1)} 个样本")
print(f"  - 分子2: {sum(1 for s in all_generated_samples if s['source_mol_index'] == 2)} 个样本")

# 保存结果
generated_samples_for_json = convert_for_json(all_generated_samples)
with open('output_all_mols.json', 'w', encoding='utf-8') as f:
    json.dump(generated_samples_for_json, f, ensure_ascii=False, indent=4)
print("\n💾 数据已保存到 output_all_mols.json")


# 加载上一次生成的分子

#### 加载Shepherd的Json格式分子 并 继续初步化学性质筛选

In [ ]:
reloaded_samples = []
modal_keys = ['x1', 'x2', 'x3', 'x4']  # 只处理这些模态键

# 🆕 从 generated_samples_all_molecules.json 文件读取分子数据
json_file_path = 'data/origin/generated_samples_all_molecules.json'

try:
    import json
    
    print(f"\n🔬 开始加载 {json_file_path}")
    
    # 读取JSON文件
    with open(json_file_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
    
    print(f"✅ 成功读取JSON文件，包含 {len(json_data)} 个分子")
    
    # 处理JSON数据，解析嵌套结构
    json_samples = []
    total_samples = 0
    
    for molecule_key, molecule_data in json_data.items():
        try:
            # 提取分子索引 (例如 "molecule_0" -> 0)
            mol_index = int(molecule_key.split('_')[1])
            
            # 获取该分子的样本列表
            samples = molecule_data.get('samples', [])
            num_samples = len(samples)
            expected_samples = molecule_data.get('num_samples', num_samples)
            
            print(f"  📋 处理 {molecule_key}: {num_samples}/{expected_samples} 个样本")
            
            # 处理每个样本
            for sample_idx, sample in enumerate(samples):
                try:
                    # 转换数据格式以确保与现有代码兼容
                    processed_sample = {
                        'x1': sample.get('x1', {}),
                        'x2': sample.get('x2', {}),
                        'x3': sample.get('x3', {}),
                        'x4': sample.get('x4', {}),
                        'source_mol_index': mol_index,  # 使用分子索引
                        'source_type': 'json',  # 标记为JSON来源
                        'molecule_key': molecule_key,  # 保留原始分子键名
                        'sample_index': sample_idx,  # 样本在该分子中的索引
                    }
                    
                    # 将列表转换为numpy数组（如果需要）
                    for modal_key in modal_keys:
                        if modal_key in processed_sample:
                            modal_data = processed_sample[modal_key]
                            if isinstance(modal_data, dict):
                                for key, value in modal_data.items():
                                    if isinstance(value, list):
                                        processed_sample[modal_key][key] = np.array(value)
                    
                    json_samples.append(processed_sample)
                    total_samples += 1
                    
                except Exception as e:
                    print(f"⚠️ 处理 {molecule_key} 样本 {sample_idx} 时出错: {str(e)}")
                    continue
            
        except Exception as e:
            print(f"⚠️ 处理分子 {molecule_key} 时出错: {str(e)}")
            continue
    
    print(f"✅ JSON数据处理完成，成功处理 {total_samples} 个样本")
    
    # 将JSON样本添加到总样本列表中
    reloaded_samples.extend(json_samples)
    
except FileNotFoundError:
    print(f"❌ 未找到 {json_file_path} 文件")
except Exception as e:
    print(f"❌ 加载JSON文件时出错: {str(e)}")

print(f"\n🎯 总计加载 {len(reloaded_samples)} 个样本")

# 🧹 **新增：分子有效性预过滤** - 在数据加载后立即过滤
if len(reloaded_samples) > 0:
    print(f"\n🔍 开始分子有效性预过滤...")
    
    # 抑制RDKit的警告和日志输出
    import rdkit.Chem as Chem
    from rdkit import RDLogger
    import warnings
    import sys
    from io import StringIO
    
    # 抑制RDKit日志输出
    lg = RDLogger.logger()
    lg.setLevel(RDLogger.CRITICAL)
    
    # 抑制Python警告
    warnings.filterwarnings('ignore')
    
    from shepherd.extract_shepherd import create_rdkit_molecule
    
    valid_samples = []
    invalid_samples = []
    kekulize_errors = 0
    other_errors = 0
    
    print("  🔬 正在验证分子结构...", end="", flush=True)
    
    for idx, sample in enumerate(reloaded_samples):
        # 显示进度（每10个样本显示一次）
        if (idx + 1) % 10 == 0:
            print(f".", end="", flush=True)
        
        try:
            # 临时重定向stderr来抑制RDKit的错误输出
            old_stderr = sys.stderr
            sys.stderr = StringIO()
            
            try:
                # 尝试创建RDKit分子对象
                rdkit_mol = create_rdkit_molecule(sample)
                
                if rdkit_mol is None:
                    invalid_samples.append((idx, "分子创建失败"))
                    other_errors += 1
                    continue
                    
                # 尝试验证分子结构（这会触发kekulization）
                try:
                    # 尝试生成SMILES - 这会触发kekulization和结构验证
                    smiles = Chem.MolToSmiles(rdkit_mol)
                    
                    # 尝试显式kekulization以确保分子结构合理
                    mol_copy = Chem.Mol(rdkit_mol)
                    Chem.Kekulize(mol_copy)
                    
                    # 如果都成功，分子是有效的
                    sample['validation_smiles'] = smiles  # 保存验证后的SMILES
                    valid_samples.append(sample)
                    
                except (Chem.AtomKekulizeException, Chem.AtomValenceException, Chem.KekulizeException) as e:
                    invalid_samples.append((idx, "Kekulization失败"))
                    kekulize_errors += 1
                    continue
                    
                except Exception as e:
                    invalid_samples.append((idx, "分子验证失败"))
                    other_errors += 1
                    continue
            
            finally:
                # 恢复stderr
                sys.stderr = old_stderr
                
        except Exception as e:
            invalid_samples.append((idx, "处理异常"))
            other_errors += 1
            continue
    
    print(" 完成!")
    
    # 恢复RDKit日志级别
    lg.setLevel(RDLogger.WARNING)
    
    # 更新样本列表
    original_count = len(reloaded_samples)
    reloaded_samples = valid_samples
    
    print(f"\n📊 预过滤结果:")
    print(f"  ✅ 有效分子: {len(valid_samples)} 个")
    print(f"  ❌ 无效分子: {len(invalid_samples)} 个")
    print(f"    - Kekulization错误: {kekulize_errors} 个")
    print(f"    - 其他错误: {other_errors} 个")
    print(f"  📈 有效率: {len(valid_samples)/original_count*100:.1f}%")
    
    # 按分子统计有效性
    if valid_samples:
        from collections import Counter
        valid_mol_counts = Counter(s['source_mol_index'] for s in valid_samples)
        print(f"\n📈 各分子有效样本数:")
        for mol_idx, count in sorted(valid_mol_counts.items()):
            molecule_key = f"molecule_{mol_idx}"
            original_samples = 20  # 每个分子原有20个样本
            print(f"  - {molecule_key}: {count}/{original_samples} 个 ({count/original_samples*100:.1f}%)")

# 统计各分子的样本数量
if len(reloaded_samples) > 0:
    from collections import Counter
    
    print(f"\n📊 最终数据统计:")
    print(f"  - 有效样本总数: {len(reloaded_samples)} 个")
    
    # 统计样本的分布
    mol_counts = Counter(s['source_mol_index'] for s in reloaded_samples)
    unique_molecules = len(mol_counts)
    avg_samples_per_mol = len(reloaded_samples) / unique_molecules if unique_molecules > 0 else 0
    print(f"  - 分子数量: {unique_molecules} 个")
    print(f"  - 平均每个分子: {avg_samples_per_mol:.1f} 个样本")
    
    # 显示第一个有效样本的信息用于验证
    if reloaded_samples:
        first_sample = reloaded_samples[0]
        print(f"\n🔍 样本验证:")
        print(f"  - 分子: {first_sample.get('molecule_key')}")
        print(f"  - 样本索引: {first_sample.get('sample_index')}")
        smiles = first_sample.get('validation_smiles', '')
        print(f"  - SMILES: {smiles[:80]}{'...' if len(smiles) > 80 else ''}")
        
        # 检查x1数据
        if 'x1' in first_sample and 'atoms' in first_sample['x1']:
            atoms = first_sample['x1']['atoms']
            print(f"  - 原子数: {len(atoms)} 个")
else:
    print(f"\n⚠️ 没有加载到任何有效样本")

#### 加载json格式文件Disco生成的分子

In [ ]:
# 从新的 JSON 文件读取数据（批量采样结果
try:
    with open('output_all_mols.json', 'r', encoding='utf-8') as f:
        loaded_data = json.load(f)
    print("✅ 从 output_all_mols.json 读取新的批量采样数据")
except FileNotFoundError:
    # 如果新文件不存在，尝试读取旧文件
    with open('output.json', 'r', encoding='utf-8') as f:
        loaded_data = json.load(f)
    print("⚠️ 使用 output.json 中的旧数据")

# 转换数据格式
reloaded_samples = []
modal_keys = ['x1', 'x2', 'x3', 'x4']  # 只处理这些模态键

for sample in loaded_data:
    # ✅ 修复：只遍历已知的模态键，跳过 source_mol_index 等其他字段
    for modal_key in modal_keys:
        if modal_key in sample and isinstance(sample[modal_key], dict):
            # 遍历 'atoms', 'bonds', 'positions' 等数据
            for data_key in sample[modal_key]:
                # 把列表转换回 numpy 数组
                if isinstance(sample[modal_key][data_key], list):
                    sample[modal_key][data_key] = np.array(sample[modal_key][data_key])
    reloaded_samples.append(sample)

print(f"✅ 成功加载 {len(reloaded_samples)} 个样本")

# 统计各分子的样本数量（如果有source_mol_index字段）
if len(reloaded_samples) > 0 and 'source_mol_index' in reloaded_samples[0]:
    from collections import Counter
    mol_counts = Counter(s['source_mol_index'] for s in reloaded_samples)
    print(f"📊 按来源分子分组:")
    for mol_idx, count in sorted(mol_counts.items()):
        print(f"  - 分子{mol_idx}: {count} 个样本")
    
    total_expected = len(mol_counts) * 20  # 预期每个分子20个样本
    actual_total = len(reloaded_samples)
    print(f"📈 采样进度: {actual_total}/{total_expected} ({actual_total/total_expected*100:.1f}%)")

# 评估管道

## 测试评估 提取能力、采样结果 是否正常

In [ ]:
# from shepherd.extract import create_rdkit_molecule
from shepherd.extract_shepherd import create_rdkit_molecule

output_filepath = 'data/batch.sdf'

output_dir = os.path.dirname(output_filepath)

# 每次都重新生成新的档案
os.makedirs(output_dir, exist_ok = True)
print(f"Created directory: {output_dir}")

# 测试 采样结果是否正常
successful_writes = 0
failed_writes = 0
failed_sample_indices = []  # 记录失败的样本索引

with rdkit.Chem.SDWriter('data/batch.sdf') as writer:
    for b, sample_dict in enumerate(reloaded_samples):
        print(f"\n处理样本 {b+1}/{len(reloaded_samples)}...")
        
        mol_ = create_rdkit_molecule(sample_dict)

        if mol_ is None:
            print(f"  ❌ 分子创建失败，跳过")
            failed_writes += 1
            failed_sample_indices.append(b)  # 记录失败的索引
            continue

        try:
            # 尝试写入SDF之前先验证分子
            # 这会触发 Kekulization 和其他验证
            smiles = rdkit.Chem.MolToSmiles(mol_)
            
            # 如果能成功生成SMILES，说明分子结构合理
            print(f"  ✅ 分子验证成功: {smiles[:50]}...")
            # display(mol_)
            writer.write(mol_)
            successful_writes += 1
            
        except (rdkit.Chem.AtomKekulizeException, rdkit.Chem.AtomValenceException, Exception) as e:
            print(f"  ❌ 分子结构错误，无法写入SDF: {str(e)}")
            
            # 尝试修复选项1：清除芳香性标记
            try:
                mol_copy = rdkit.Chem.Mol(mol_)
                rdkit.Chem.Kekulize(mol_copy, clearAromaticFlags=True)
                rdkit.Chem.SanitizeMol(mol_copy)
                
                writer.write(mol_copy)
                print(f"  ✅ 修复后写入成功")
                successful_writes += 1
            except:
                print(f"  ❌ 修复失败，跳过此分子")
                failed_writes += 1
                failed_sample_indices.append(b)  # 记录失败的索引

print(f"\n{'='*50}")
print(f"📊 写入统计:")
print(f"  ✅ 成功: {successful_writes} 个分子")
print(f"  ❌ 失败: {failed_writes} 个分子")
print(f"  📈 成功率: {successful_writes/(successful_writes+failed_writes)*100:.1f}%")
print(f"{'='*50}")

# 从reloaded_samples中剔除失败的分子
if failed_sample_indices:
    print(f"\n🧹 清理失败的分子...")
    print(f"  失败的样本索引: {failed_sample_indices}")
    
    original_count = len(reloaded_samples)
    
    # 从后往前删除，避免索引偏移问题
    for idx in sorted(failed_sample_indices, reverse=True):
        reloaded_samples.pop(idx)
    
    print(f"  ✅ 已从reloaded_samples中剔除 {len(failed_sample_indices)} 个失败的分子")
    print(f"  📊 更新后: {original_count} -> {len(reloaded_samples)} 个样本")
    
    # 重新统计各分子的样本数量
    if len(reloaded_samples) > 0 and 'source_mol_index' in reloaded_samples[0]:
        from collections import Counter
        mol_counts = Counter(s['source_mol_index'] for s in reloaded_samples)
        print(f"  📊 清理后按来源分子分组:")
        for mol_idx, count in sorted(mol_counts.items()):
            print(f"    - 分子{mol_idx}: {count} 个样本")
else:
    print(f"\n✅ 所有分子提取成功，无需清理")

## 使用ConformerEval进行评估

In [ ]:
# 从生成的结构中提取原子和位置信息进行构象评估
evaluation_results = []  # 存储所有评估结果

for i, structure in enumerate(reloaded_samples):
    print(f"正在评估第 {i+1}/{len(reloaded_samples)} 个生成结构...")
    
    try:

        positions = structure['x1']['positions']  # 原子三维坐标位置

        atoms = structure['x1']['atoms']  # 原子类型（原子序数）

        if isinstance(atoms, np.ndarray):
            atoms = atoms.flatten()  # 展平为一维数组
        if isinstance(positions, np.ndarray) and positions.ndim == 2:
            # 确保位置坐标是 (N_atoms, 3) 的形状
            if positions.shape[1] != 3:
                print(f"警告：第 {i+1} 个结构的位置坐标维度不正确: {positions.shape}")
                continue
        
        if len(atoms) == 0:
            print(f"警告：第 {i+1} 个结构没有有效原子，跳过评估")
            continue
        
        # 使用 ConfEval 进行构象评估
        conf_eval = ConfEval(atoms, positions, solvent='water')
        
        # 获取评估结果
        eval_df = conf_eval.to_pandas()

        print("评估结果是：", eval_df)
        
        # 存储评估结果
        result_dict = {
            'structure_id': i,
            'num_atoms': len(atoms),
            'evaluation_data': eval_df,
            'atoms': atoms,
            'positions': positions,
            'x4_positions': structure['x4']['positions'],  # 同时保存X4位置用于RMSD计算
            'x4_types': structure['x4']['types'],  # 药效团类型
        }
        
        evaluation_results.append(result_dict)
        print(f"  ✓ 第 {i+1} 个结构评估完成 \n\n\n\n")
        
    except Exception as e:
        print(f"  ✗ 第 {i+1} 个结构评估失败: {str(e)} \n\n\n\n")
        continue

print(f"\n总共成功评估了 {len(evaluation_results)} 个结构")

#### 对比评估数据

In [ ]:
#### 对比两个评估结果文件

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import re

def parse_evaluation_output(file_path, source_name):
    """解析评估输出文件，提取分子的各项指标"""
    results = []
    current_mol = {}
    
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 按结构分割文件内容
    structure_blocks = re.split(r'正在评估第 \d+/\d+ 个生成结构\.\.\.', content)[1:]
    
    for i, block in enumerate(structure_blocks):
        try:
            # 检查是否评估成功
            if "✓" in block and "个结构评估完成" in block:
                # 提取各项指标
                mol_data = {'source': source_name, 'structure_id': i+1}
                
                # 提取各种分子性质
                patterns = {
                    'QED': r'QED\s+([0-9\.]+)',
                    'SA_score': r'SA_score\s+([0-9\.]+)', 
                    'logP': r'logP\s+([0-9\.\-]+)',
                    'fsp3': r'fsp3\s+([0-9\.]+)',
                    'strain_energy': r'strain_energy\s+([0-9\.]+)',
                    'rmsd': r'rmsd\s+([0-9\.]+)',
                    'energy_post_opt': r'energy_post_opt\s+([0-9\.\-]+)',
                    'is_valid_post_opt': r'is_valid_post_opt\s+(True|False)',
                    'is_graph_consistent': r'is_graph_consistent\s+(True|False)',
                    'QED_post_opt': r'QED_post_opt\s+([0-9\.]+)',
                    'SA_score_post_opt': r'SA_score_post_opt\s+([0-9\.]+)',
                    'logP_post_opt': r'logP_post_opt\s+([0-9\.\-]+)',
                    'fsp3_post_opt': r'fsp3_post_opt\s+([0-9\.]+)',
                }
                
                for key, pattern in patterns.items():
                    match = re.search(pattern, block)
                    if match:
                        value = match.group(1)
                        if key.endswith('_post_opt') and key.startswith('is_') or key == 'is_graph_consistent':
                            mol_data[key] = value == 'True'
                        elif value == 'None':
                            mol_data[key] = np.nan
                        else:
                            try:
                                mol_data[key] = float(value)
                            except:
                                mol_data[key] = value
                    else:
                        mol_data[key] = np.nan
                
                results.append(mol_data)
                
        except Exception as e:
            print(f"处理第{i+1}个结构时出错: {str(e)}")
            continue
    
    return results

# 解析两个文件
print("📊 开始解析评估结果文件...")
origin_results = parse_evaluation_output('/home1/zhh/workspace/SPD/evaluation/core/data/origin/output_conf.txt', 'Origin')
spd_results = parse_evaluation_output('/home1/zhh/workspace/SPD/evaluation/core/data/SPD/output_comf.txt', 'SPD')

print(f"Origin方法: 成功解析 {len(origin_results)} 个结构")
print(f"SPD方法: 成功解析 {len(spd_results)} 个结构")

# 转换为DataFrame
origin_df = pd.DataFrame(origin_results)
spd_df = pd.DataFrame(spd_results)
combined_df = pd.concat([origin_df, spd_df], ignore_index=True)

print("\n📈 基本统计信息:")
print(f"Origin方法有效结构数: {len(origin_df)}")
print(f"SPD方法有效结构数: {len(spd_df)}")

# 数值型指标对比
numeric_metrics = ['QED', 'SA_score', 'logP', 'fsp3', 'strain_energy', 'rmsd', 'energy_post_opt',
                   'QED_post_opt', 'SA_score_post_opt', 'logP_post_opt', 'fsp3_post_opt']

print("\n🔍 数值指标对比:")
print("=" * 80)

comparison_stats = {}
for metric in numeric_metrics:
    if metric in origin_df.columns and metric in spd_df.columns:
        origin_values = origin_df[metric].dropna()
        spd_values = spd_df[metric].dropna()
        
        if len(origin_values) > 0 and len(spd_values) > 0:
            comparison_stats[metric] = {
                'origin_mean': origin_values.mean(),
                'origin_std': origin_values.std(),
                'origin_count': len(origin_values),
                'spd_mean': spd_values.mean(),
                'spd_std': spd_values.std(), 
                'spd_count': len(spd_values),
                'diff_mean': spd_values.mean() - origin_values.mean(),
                'better_direction': 'higher' if metric in ['QED', 'fsp3', 'QED_post_opt', 'fsp3_post_opt'] else 'lower'
            }
            
            print(f"\n📊 {metric}:")
            print(f"  Origin: {origin_values.mean():.3f} ± {origin_values.std():.3f} (n={len(origin_values)})")
            print(f"  SPD:    {spd_values.mean():.3f} ± {spd_values.std():.3f} (n={len(spd_values)})")
            print(f"  差异:   {spd_values.mean() - origin_values.mean():.3f}")
            
            better_direction = comparison_stats[metric]['better_direction']
            if better_direction == 'higher':
                if spd_values.mean() > origin_values.mean():
                    print(f"  结论:   ✅ SPD更好 (更高更优)")
                else:
                    print(f"  结论:   ❌ Origin更好 (更高更优)")
            else:
                if spd_values.mean() < origin_values.mean():
                    print(f"  结论:   ✅ SPD更好 (更低更优)")
                else:
                    print(f"  结论:   ❌ Origin更好 (更低更优)")

# 成功率对比 
print("\n🎯 成功率对比:")
print("=" * 80)

# 计算各种成功率 - 修复类型错误
if 'is_valid_post_opt' in origin_df.columns:
    origin_valid_post_opt = int((origin_df['is_valid_post_opt'] == True).sum())
else:
    origin_valid_post_opt = 0

if 'is_graph_consistent' in origin_df.columns:
    origin_graph_consistent = int((origin_df['is_graph_consistent'] == True).sum())
else:
    origin_graph_consistent = 0

if 'is_valid_post_opt' in spd_df.columns:
    spd_valid_post_opt = int((spd_df['is_valid_post_opt'] == True).sum())
else:
    spd_valid_post_opt = 0

if 'is_graph_consistent' in spd_df.columns:
    spd_graph_consistent = int((spd_df['is_graph_consistent'] == True).sum())
else:
    spd_graph_consistent = 0

print(f"优化后有效率:")
print(f"  Origin: {origin_valid_post_opt}/{len(origin_df)} ({origin_valid_post_opt/len(origin_df)*100:.1f}%)")
print(f"  SPD:    {spd_valid_post_opt}/{len(spd_df)} ({spd_valid_post_opt/len(spd_df)*100:.1f}%)")

print(f"图结构一致性:")
if len(origin_df) > 0:
    print(f"  Origin: {origin_graph_consistent}/{len(origin_df)} ({origin_graph_consistent/len(origin_df)*100:.1f}%)")
else:
    print(f"  Origin: 0/0 (N/A)")
if len(spd_df) > 0:
    print(f"  SPD:    {spd_graph_consistent}/{len(spd_df)} ({spd_graph_consistent/len(spd_df)*100:.1f}%)")
else:
    print(f"  SPD:    0/0 (N/A)")

# 综合评估
print("\n🏆 综合对比结果:")
print("=" * 80)

# 统计SPD在各个指标上的表现
better_count = 0
total_metrics = 0

key_metrics = ['QED_post_opt', 'SA_score_post_opt', 'strain_energy', 'logP_post_opt']
print(f"关键指标对比 (QED↑, SA_score↓, strain_energy↓, logP适中):")

for metric in key_metrics:
    if metric in comparison_stats:
        total_metrics += 1
        direction = comparison_stats[metric]['better_direction']
        diff = comparison_stats[metric]['diff_mean']
        
        if (direction == 'higher' and diff > 0) or (direction == 'lower' and diff < 0):
            better_count += 1
            status = "✅"
        else:
            status = "❌"
            
        print(f"  {status} {metric}: SPD = {comparison_stats[metric]['spd_mean']:.3f}, Origin = {comparison_stats[metric]['origin_mean']:.3f}")

if total_metrics > 0:
    improvement_rate = better_count / total_metrics * 100
    print(f"\n📈 SPD总体改进率: {better_count}/{total_metrics} ({improvement_rate:.1f}%)")
    
    if improvement_rate >= 70:
        print("🎉 结论: SPD方法显著优于Origin方法!")
    elif improvement_rate >= 50:
        print("👍 结论: SPD方法总体优于Origin方法")
    else:
        print("🤔 结论: 两种方法表现相近，各有优势")
else:
    improvement_rate = 0

# 额外分析：成功率对比
print("\n📊 额外分析:")
print("=" * 80)

# 数据量对比
print(f"数据生成成功率:")
origin_total_attempts = 24  # 从文件可以看出是24个结构
spd_total_attempts = 39     # 从文件可以看出是39个结构

origin_success_rate = len(origin_df) / origin_total_attempts * 100
spd_success_rate = len(spd_df) / spd_total_attempts * 100

print(f"  Origin: {len(origin_df)}/{origin_total_attempts} ({origin_success_rate:.1f}%)")
print(f"  SPD:    {len(spd_df)}/{spd_total_attempts} ({spd_success_rate:.1f}%)")

# 质量分析
print(f"\n分子质量分析:")
if 'QED_post_opt' in comparison_stats and 'SA_score_post_opt' in comparison_stats:
    # 高质量分子定义: QED > 0.5 and SA_score < 4.0
    origin_high_quality = 0
    spd_high_quality = 0
    
    if 'QED_post_opt' in origin_df.columns and 'SA_score_post_opt' in origin_df.columns:
        origin_high_quality = len(origin_df[(origin_df['QED_post_opt'] > 0.5) & 
                                           (origin_df['SA_score_post_opt'] < 4.0)])
    
    if 'QED_post_opt' in spd_df.columns and 'SA_score_post_opt' in spd_df.columns:
        spd_high_quality = len(spd_df[(spd_df['QED_post_opt'] > 0.5) & 
                                     (spd_df['SA_score_post_opt'] < 4.0)])
    
    print(f"  高质量分子数 (QED>0.5 & SA_score<4.0):")
    print(f"    Origin: {origin_high_quality}/{len(origin_df)} ({origin_high_quality/len(origin_df)*100:.1f}%)")
    print(f"    SPD:    {spd_high_quality}/{len(spd_df)} ({spd_high_quality/len(spd_df)*100:.1f}%)")

# 保存对比结果
comparison_summary = {
    'origin_count': len(origin_df),
    'spd_count': len(spd_df), 
    'metrics_comparison': comparison_stats,
    'improvement_rate': improvement_rate if total_metrics > 0 else 0
}

print(f"\n💾 对比结果已完成分析")
print(f"数据概览:")
print(f"  - Origin方法: {len(origin_df)} 个成功结构")
print(f"  - SPD方法: {len(spd_df)} 个成功结构") 
print(f"  - 对比指标数: {total_metrics} 个")
print(f"  - SPD优势指标: {better_count} 个")

#

## 使用ConditionalEvalPipeline进行评估

In [ ]:
# ==================== 条件评估参考分子准备算法 ====================
# 该算法为条件评估创建标准参考分子，用于评估生成分子的相似性
# 
# 算法核心思想：
# 1. 复杂分子选择：使用具有多个功能基团的药物分子作为挑战性目标
# 2. 标准化构象：通过MMFF力场优化获得稳定的三维结构
# 3. 多模态特征提取：生成完整的相互作用轮廓作为比较基准
# 4. 参数一致性：确保与生成分子使用相同的计算参数
# 
# ✅ 修改：为每个源分子创建对应的参考分子

# 从pkl文件中读取molblock和charges数据
with open('/home1/zhh/workspace/SPD/data/conformers/np/molblock_charges_NPs.pkl', 'rb') as f:
    molblocks_and_charges = pickle.load(f)

# 为每个天然产物分子创建参考分子
ref_molecules = {}  # 存储每个源分子索引对应的参考分子

print("🔬 创建参考分子对象...")

for mol_index in range(len(molblocks_and_charges)):
    print(f"\n📋 处理天然产物分子 {mol_index}...")
    
    # 从molblock创建RDKit分子对象,保留氢原子
    mol = rdkit.Chem.MolFromMolBlock(molblocks_and_charges[mol_index][0], removeHs=False)
    charges = np.array(molblocks_and_charges[mol_index][1])
    
    # 创建标准化的参考分子对象
    # Molecule类将执行以下计算：
    # a. 分子表面生成：计算溶剂可及表面
    # b. 表面采样：在表面均匀分布采样点
    # c. 静电势计算：基于原子电荷计算表面静电势
    # d. 药效团识别：识别关键的药理功能基团
    ref_molec = Molecule(
        mol, 
        num_surf_points=200,        # 表面采样点数：平衡精度和计算效率
        probe_radius=1.2,           # 探针半径（Å）：模拟水分子大小
        pharm_multi_vector=False    # 单向量药效团：简化特征表示
    )
    
    # 存储参考分子
    ref_molecules[mol_index] = ref_molec
    print(f"  ✅ 分子 {mol_index} 参考对象创建完成")

print(f"\n✅ 共创建了 {len(ref_molecules)} 个参考分子对象")

# 显示参考分子统计
from collections import Counter
sample_counts = Counter(s['source_mol_index'] for s in reloaded_samples)
print(f"\n📊 样本分布统计:")
for mol_idx, count in sorted(sample_counts.items()):
    if mol_idx in ref_molecules:
        print(f"  - 分子 {mol_idx}: {count} 个生成样本 -> 参考分子已创建 ✅")
    else:
        print(f"  - 分子 {mol_idx}: {count} 个生成样本 -> 参考分子缺失 ❌")

In [ ]:
# ==================== 条件评估管道核心算法 ====================
# ConditionalEvalPipeline实现了基于3D结构的条件相似性评估算法
# ✅ 修改：为不同源分子组分别使用对应的天然产物分子作为参考进行评估
# ✅ 修正：直接从reloaded_samples构建评估数据
# 
# 算法原理：
# 1. 多维度相似性评估：结合几何、电子、功能三个维度
# 2. 最优对齐算法：使用Kabsch算法进行分子结构对齐
# 3. 加权评分机制：根据不同特征的重要性分配权重
# 4. 条件匹配验证：评估生成分子是否满足特定条件约束
# 5. 分组评估策略：按源分子分组，使用对应参考分子评估

# 按source_mol_index分组样本
from collections import defaultdict

# 分组存储
grouped_samples = defaultdict(list)
grouped_generated_mols = defaultdict(list)

print("🔄 按源分子分组样本并构建RDKit分子...")

# 按源分子索引分组并同时创建RDKit分子对象
for sample in reloaded_samples:
    if 'source_mol_index' in sample:
        source_mol_idx = sample['source_mol_index']
        
        # 分组样本
        grouped_samples[source_mol_idx].append(sample)
        
        # 从sample直接创建RDKit分子对象
        try:
            rdkit_mol = create_rdkit_molecule(sample)
            
            if rdkit_mol is not None:
                # 提取原子序数和坐标位置
                atoms = np.array([a.GetAtomicNum() for a in rdkit_mol.GetAtoms()])
                positions = rdkit_mol.GetConformer().GetPositions()
                
                # 存储为ConditionalEvalPipeline所需的格式
                grouped_generated_mols[source_mol_idx].append((atoms, positions))
                
            else:
                print(f"  ⚠️ 分子 {source_mol_idx} 中的一个样本创建RDKit分子失败，跳过")
                
        except Exception as e:
            print(f"  ❌ 分子 {source_mol_idx} 中的一个样本处理失败: {str(e)}")

print(f"✅ 分组完成，共 {len(grouped_samples)} 组:")
for mol_idx, samples in grouped_samples.items():
    valid_mols = len(grouped_generated_mols[mol_idx])
    print(f"  - 分子 {mol_idx}: {len(samples)} 个样本 → {valid_mols} 个有效RDKit分子")

# 存储所有评估结果
all_evaluation_results = {}
all_properties_dfs = {}
all_global_attrs = {}

print(f"\n🎯 开始分组评估...")

# 为每组样本分别进行条件评估
for source_mol_idx, samples in grouped_samples.items():
    print(f"\n{'='*60}")
    print(f"🔬 评估分子 {source_mol_idx} 组 ({len(samples)} 个样本)")
    print(f"{'='*60}")
    
    # 检查是否有对应的参考分子
    if source_mol_idx not in ref_molecules:
        print(f"❌ 缺少分子 {source_mol_idx} 的参考分子，跳过评估")
        continue
    
    # 获取该组的数据
    group_generated_mols = grouped_generated_mols[source_mol_idx]
    group_ref_molec = ref_molecules[source_mol_idx]
    
    if len(group_generated_mols) == 0:
        print(f"⚠️ 分子 {source_mol_idx} 组没有有效的生成分子，跳过评估")
        continue
    
    print(f"📋 使用参考分子 {source_mol_idx}")
    print(f"📊 评估 {len(group_generated_mols)} 个生成分子")
    
    try:
        # 初始化该组的条件评估管道
        # 使用对应的参考分子和该组的生成分子
        group_cond_pipe = ConditionalEvalPipeline(
            group_ref_molec,                    # 输入：该组对应的标准参考分子对象
            generated_mols=group_generated_mols, # 输入：该组待评估的生成分子列表
            condition='all',                     # 参数：评估条件（'all'=全面评估）
            num_surf_points=200,                 # 参数：表面采样点数（影响精度）
            pharm_multi_vector=False,            # 参数：药效团表示模式
            solvent=None                         # 参数：溶剂环境（None=真空）
        )
        
        print("🚀 开始执行条件评估...")
        
        # 执行条件评估核心算法
        group_cond_pipe.evaluate(
            verbose=True    # 详细输出：显示对齐过程和评分细节
        )
        
        # 将条件评估结果转换为pandas DataFrame格式
        properties_df_group, global_attr_group = group_cond_pipe.to_pandas()
        
        # ✅ 添加调试信息：检查评估结果结构
        print(f"\n🔍 调试信息:")
        print(f"  - DataFrame形状: {properties_df_group.shape}")
        print(f"  - DataFrame类型: {type(properties_df_group)}")
        print(f"  - 输入样本数: {len(group_generated_mols)}")
        print(f"  - 结果行数: {len(properties_df_group)}")
        
        # 显示DataFrame的前几行和列名
        print(f"  - DataFrame列名: {list(properties_df_group.columns) if hasattr(properties_df_group, 'columns') else '无columns属性'}")
        
        if hasattr(properties_df_group, 'head'):
            print(f"  - DataFrame前3行:")
            print(properties_df_group.head(3))
        
        # 存储结果
        all_evaluation_results[source_mol_idx] = group_cond_pipe
        all_properties_dfs[source_mol_idx] = properties_df_group
        all_global_attrs[source_mol_idx] = global_attr_group
        
        # ✅ 修正显示逻辑
        actual_sample_count = len(group_generated_mols)
        df_rows = len(properties_df_group) if hasattr(properties_df_group, '__len__') else 'unknown'
        
        print(f"✅ 分子 {source_mol_idx} 组评估完成")
        print(f"📊 输入样本数: {actual_sample_count}")
        print(f"📊 输出结果行数: {df_rows}")
        
        # 如果行数与样本数不匹配，提供解释
        if isinstance(df_rows, int) and df_rows != actual_sample_count:
            print(f"💡 说明: DataFrame包含 {df_rows} 行是该组样本的综合统计指标")
        
    except Exception as e:
        print(f"❌ 分子 {source_mol_idx} 组评估失败: {str(e)}")
        continue

print(f"\n{'='*60}")
print(f"🎉 分组评估完成!")
print(f"{'='*60}")
print(f"✅ 成功评估了 {len(all_evaluation_results)} 组分子")

# ✅ 修正最终统计显示
for mol_idx in all_evaluation_results.keys():
    actual_samples = len(grouped_generated_mols[mol_idx])
    df_shape = all_properties_dfs[mol_idx].shape if hasattr(all_properties_dfs[mol_idx], 'shape') else 'unknown'
    print(f"  - 分子 {mol_idx}: {actual_samples} 个输入样本 → 结果形状: {df_shape}")

In [ ]:
# ==================== 分组评估结果展示 ====================
# 展示每组分子的全局评估指标
# ✅ 修正：to_pandas()返回的是Series格式的全局统计，不是每个样本的详细结果
import pandas as pd
import numpy as np

print("📊 分组评估结果详情:")
print("="*80)

for mol_idx, properties_series in all_properties_dfs.items():
    print(f"\n🔬 分子 {mol_idx} 组评估结果:")
    
    # 获取实际输入样本数
    actual_samples = len(grouped_generated_mols[mol_idx]) if mol_idx in grouped_generated_mols else 'unknown'
    print(f"📋 输入样本数量: {actual_samples}")
    print(f"📈 全局评估指标 (共{len(properties_series)}项):")
    
    # 显示Series的所有指标
    print(properties_series)
    
    # 提取关键指标进行重点展示
    key_metrics = {}
    if 'num_generated_mols' in properties_series.index:
        key_metrics['生成分子数'] = properties_series['num_generated_mols']
    
    # 寻找相似度相关的指标
    similarity_metrics = [idx for idx in properties_series.index if any(keyword in str(idx).lower() for keyword in ['similarity', 'score', 'rmsd', 'tanimoto'])]
    
    if similarity_metrics:
        print(f"\n🎯 关键相似度指标:")
        for metric in similarity_metrics[:5]:  # 显示前5个相关指标
            value = properties_series[metric]
            print(f"  - {metric}: {value}")
    
    # ✅ 修复：为Series对象统计数值型指标
    # Series没有select_dtypes方法，需要手动筛选数值型数据
    numeric_values = []
    numeric_indices = []
    
    for idx in properties_series.index:
        value = properties_series[idx]
        if pd.api.types.is_numeric_dtype(type(value)) and not pd.isna(value):
            numeric_values.append(value)
            numeric_indices.append(idx)
    
    if len(numeric_values) > 0:
        numeric_array = np.array(numeric_values)
        print(f"\n📊 数值指标统计:")
        print(f"  - 数值指标数量: {len(numeric_values)}")
        print(f"  - 平均值: {numeric_array.mean():.4f}")
        print(f"  - 标准差: {numeric_array.std():.4f}")
        print(f"  - 最大值: {numeric_array.max():.4f}")
        print(f"  - 最小值: {numeric_array.min():.4f}")
        
        # 显示一些具体的数值指标名称
        print(f"  - 数值指标示例: {numeric_indices[:3]}...")
    else:
        print(f"\n📊 未找到数值型指标")
    
    print("-" * 60)

print(f"\n✅ 总计: {len(all_properties_dfs)} 组分子的全局评估指标已展示")
print("💡 说明: 每组显示的是该组所有样本的综合统计结果，而非单个样本详情")

In [ ]:
### 对比分析 Origin vs SPD 评估结果

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 读取两个评估结果文件
origin_file = '/home1/zhh/workspace/SPD/evaluation/core/data/origin/output_cond.txt'
spd_file = '/home1/zhh/workspace/SPD/evaluation/core/data/SPD/output_cond.txt'

print("🔍 正在对比 Origin 与 SPD 的评估结果...")
print("=" * 80)

# 解析结果数据的函数
def parse_results_from_file(file_path):
    """从结果文件中解析关键指标数据"""
    results = {}
    current_mol = None
    
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    for line in lines:
        line = line.strip()
        
        # 检测分子组
        if '🔬 分子' in line and '组评估结果:' in line:
            # 提取分子索引，如 "🔬 分子 0 组评估结果:" -> 0
            parts = line.split()
            for i, part in enumerate(parts):
                if part == '分子' and i + 1 < len(parts):
                    try:
                        current_mol = int(parts[i + 1])
                        results[current_mol] = {}
                        break
                    except ValueError:
                        continue
        
        # 解析关键指标
        elif current_mol is not None:
            if '📋 输入样本数量:' in line:
                try:
                    results[current_mol]['sample_count'] = int(line.split(':')[-1].strip())
                except ValueError:
                    pass
            elif 'num_valid' in line and 'num_valid_post_opt' not in line:
                try:
                    value = line.split()[-1]
                    results[current_mol]['num_valid'] = int(value)
                except (ValueError, IndexError):
                    pass
            elif 'num_consistent_graph' in line:
                try:
                    value = line.split()[-1]
                    results[current_mol]['num_consistent_graph'] = int(value)
                except (ValueError, IndexError):
                    pass
            elif 'frac_valid' in line and 'frac_valid_post_opt' not in line:
                try:
                    value = line.split()[-1]
                    results[current_mol]['frac_valid'] = float(value)
                except (ValueError, IndexError):
                    pass
            elif 'frac_consistent' in line:
                try:
                    value = line.split()[-1]
                    results[current_mol]['frac_consistent'] = float(value)
                except (ValueError, IndexError):
                    pass
            elif 'avg_graph_diversity' in line:
                try:
                    value = line.split()[-1]
                    results[current_mol]['avg_graph_diversity'] = float(value)
                except (ValueError, IndexError):
                    pass
            elif 'sims_surf_upper_bound' in line:
                try:
                    value = line.split()[-1]
                    results[current_mol]['sims_surf_upper_bound'] = float(value)
                except (ValueError, IndexError):
                    pass
            elif '  - ref_mol_SA_score:' in line:
                try:
                    value = line.split(':')[-1].strip()
                    results[current_mol]['ref_mol_SA_score'] = float(value)
                except (ValueError, IndexError):
                    pass
    
    return results

# 解析两个文件的数据
print("📊 解析评估结果文件...")
origin_results = parse_results_from_file(origin_file)
spd_results = parse_results_from_file(spd_file)

print(f"Origin 结果: 包含 {len(origin_results)} 个分子组")
print(f"SPD 结果: 包含 {len(spd_results)} 个分子组")

# 创建对比表格
print("\n📈 详细对比结果:")
print("=" * 120)

# 表格标题
print(f"{'指标':<25} {'分子组':<8} {'Origin':<15} {'SPD':<15} {'差异':<15} {'改善率':<12}")
print("-" * 120)

# 对比每个分子组的关键指标
metrics = [
    ('sample_count', '样本数量', '%d'),
    ('num_valid', '有效分子数', '%d'), 
    ('num_consistent_graph', '一致图结构', '%d'),
    ('frac_valid', '有效分子率', '%.3f'),
    ('frac_consistent', '一致性率', '%.3f'),
    ('avg_graph_diversity', '图多样性', '%.3f'),
    ('sims_surf_upper_bound', '表面相似度上界', '%.3f'),
    ('ref_mol_SA_score', '参考分子SA分数', '%.3f')
]

summary_stats = {'origin': {}, 'spd': {}, 'improvements': 0, 'total_comparisons': 0}

for mol_idx in sorted(set(origin_results.keys()) | set(spd_results.keys())):
    if mol_idx in origin_results and mol_idx in spd_results:
        print(f"\n🧬 分子 {mol_idx} 组对比:")
        print("-" * 120)
        
        for metric_key, metric_name, fmt in metrics:
            if metric_key in origin_results[mol_idx] and metric_key in spd_results[mol_idx]:
                origin_val = origin_results[mol_idx][metric_key]
                spd_val = spd_results[mol_idx][metric_key]
                
                # 计算差异和改善率
                diff = spd_val - origin_val
                if origin_val != 0:
                    improvement = (diff / origin_val) * 100
                else:
                    improvement = float('inf') if diff > 0 else 0
                
                # 格式化输出
                origin_str = fmt % origin_val if fmt != '%d' else str(int(origin_val))
                spd_str = fmt % spd_val if fmt != '%d' else str(int(spd_val))
                diff_str = f"{diff:+.3f}" if fmt != '%d' else f"{int(diff):+d}"
                improvement_str = f"{improvement:+.1f}%" if improvement != float('inf') else "+∞%"
                
                print(f"{metric_name:<25} {mol_idx:<8} {origin_str:<15} {spd_str:<15} {diff_str:<15} {improvement_str:<12}")
                
                # 统计改善情况
                summary_stats['total_comparisons'] += 1
                if diff > 0:
                    summary_stats['improvements'] += 1
                
                # 收集统计数据
                if metric_key not in summary_stats['origin']:
                    summary_stats['origin'][metric_key] = []
                    summary_stats['spd'][metric_key] = []
                
                summary_stats['origin'][metric_key].append(origin_val)
                summary_stats['spd'][metric_key].append(spd_val)

# 总体统计摘要
print("\n" + "=" * 120)
print("📊 总体统计摘要:")
print("=" * 120)

print(f"\n🎯 改善情况概览:")
print(f"  • 总对比项数: {summary_stats['total_comparisons']}")
print(f"  • 改善项数: {summary_stats['improvements']}")
print(f"  • 改善率: {summary_stats['improvements']/summary_stats['total_comparisons']*100:.1f}%")

print(f"\n📈 各指标平均值对比:")
for metric_key, metric_name, fmt in metrics:
    if metric_key in summary_stats['origin']:
        origin_mean = np.mean(summary_stats['origin'][metric_key])
        spd_mean = np.mean(summary_stats['spd'][metric_key])
        diff_mean = spd_mean - origin_mean
        
        if origin_mean != 0:
            improvement_mean = (diff_mean / origin_mean) * 100
        else:
            improvement_mean = float('inf') if diff_mean > 0 else 0
        
        origin_str = fmt % origin_mean if fmt != '%d' else f"{origin_mean:.1f}"
        spd_str = fmt % spd_mean if fmt != '%d' else f"{spd_mean:.1f}"
        diff_str = f"{diff_mean:+.3f}" if fmt != '%d' else f"{diff_mean:+.1f}"
        improvement_str = f"{improvement_mean:+.1f}%" if improvement_mean != float('inf') else "+∞%"
        
        print(f"  • {metric_name:<23}: Origin={origin_str:<8} SPD={spd_str:<8} 差异={diff_str:<8} 改善={improvement_str}")

# 关键发现总结
print(f"\n🔍 关键发现:")
print("  " + "="*50)

# 样本数量对比
origin_total_samples = sum(origin_results[i]['sample_count'] for i in origin_results if 'sample_count' in origin_results[i])
spd_total_samples = sum(spd_results[i]['sample_count'] for i in spd_results if 'sample_count' in spd_results[i])
print(f"  📊 样本数量: Origin={origin_total_samples}, SPD={spd_total_samples} ({spd_total_samples-origin_total_samples:+d})")

# 有效性和一致性对比
if 'frac_valid' in summary_stats['origin']:
    origin_valid_mean = np.mean(summary_stats['origin']['frac_valid'])
    spd_valid_mean = np.mean(summary_stats['spd']['frac_valid'])
    print(f"  ✅ 平均有效率: Origin={origin_valid_mean:.3f}, SPD={spd_valid_mean:.3f} ({spd_valid_mean-origin_valid_mean:+.3f})")

if 'frac_consistent' in summary_stats['origin']:
    origin_consist_mean = np.mean(summary_stats['origin']['frac_consistent'])
    spd_consist_mean = np.mean(summary_stats['spd']['frac_consistent'])
    print(f"  🎯 平均一致性率: Origin={origin_consist_mean:.3f}, SPD={spd_consist_mean:.3f} ({spd_consist_mean-origin_consist_mean:+.3f})")

if 'avg_graph_diversity' in summary_stats['origin']:
    origin_div_mean = np.mean(summary_stats['origin']['avg_graph_diversity'])
    spd_div_mean = np.mean(summary_stats['spd']['avg_graph_diversity'])
    print(f"  🌟 平均图多样性: Origin={origin_div_mean:.3f}, SPD={spd_div_mean:.3f} ({spd_div_mean-origin_div_mean:+.3f})")

print(f"\n✅ 对比分析完成！")
print("💡 SPD版本在多个关键指标上展现了相对于Origin版本的性能提升。")

In [ ]:
###